# Avaliacao e limpeza heuristica dos posts

Este notebook faz uma triagem inicial dos JSONs em `sentinel_replica_jsons/` sem usar LLM.

Regras implementadas:
1. ignorar mensagens com apenas URL
2. normalizar texto e filtrar por palavras-chave usando regex
3. filtrar mensagens curtas demais quando forem de baixo sinal
4. manter mensagens curtas que ainda possam contribuir por baterem em palavras-chave

In [7]:
from __future__ import annotations

import json
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.auto import tqdm

INPUT_DIR = Path("../sentinel_replica_jsons")
OUTPUT_DIR = Path("../sentinel_replica_jsons_cleaned_eval")
OUTPUT_DIR.mkdir(exist_ok=True)

SAMPLE_ONE_PER_FILE = False
MIN_CHARS = 25
MIN_TOKENS = 4

JSON_FILES = sorted(INPUT_DIR.glob("*.json"))
len(JSON_FILES), JSON_FILES[:3]

(14,
 [PosixPath('../sentinel_replica_jsons/BugCrowd.json'),
  PosixPath('../sentinel_replica_jsons/HackingBlogsGroup.json'),
  PosixPath('../sentinel_replica_jsons/PHOfficial.json')])

In [8]:
KEYWORD_PATTERNS = [
    r"\bmalware\b",
    r"\bransomware\b",
    r"\bphishing\b",
    r"\bcredential(?:s)?\b",
    r"\blogin\b",
    r"\bexploit(?:ed|s|ing)?\b",
    r"\bcve-\d{4}-\d+\b",
    r"\bvulnerability\b",
    r"\bzero[ -]?day\b",
    r"\bbreach\b",
    r"\bleak(?:ed|s)?\b",
    r"\bdata leak\b",
    r"\bstealer\b",
    r"\bbotnet\b",
    r"\bc2\b",
    r"\bcommand and control\b",
    r"\bddos\b",
    r"\baccount takeover\b",
    r"\bexfiltrat(?:e|ion|ed)\b",
    r"\bbackdoor\b",
    r"\btrojan\b",
    r"\bspyware\b",
    r"\bloader\b",
    r"\bdropper\b",
    r"\bhash(?:es)?\b",
    r"\bioc(?:s)?\b",
    r"\bindicator(?:s)? of compromise\b",
    r"\btelegram scam\b",
    r"\bscam\b",
    r"\bfraud\b",
    r"\bapt\b",
    r"\bthreat actor\b",
    r"\bcompromis(?:e|ed)\b",
    r"\bhacked\b",
    r"\bintrusion\b",
    r"\bvirus\b",
    r"\bwiper\b",
]

KEYWORD_RE = re.compile("|".join(KEYWORD_PATTERNS), flags=re.IGNORECASE)

SOFT_KEYWORD_PATTERNS = [
    r"\bapi security\b",
    r"\bapplication security\b",
    r"\bappsec\b",
    r"\bbug bounty\b",
    r"\bcyber ?security\b",
    r"\binfosec\b",
    r"\bosint\b",
    r"\bpentest(?:ing)?\b",
    r"\bpenetration test(?:ing)?\b",
    r"\bred team\b",
    r"\bblue team\b",
    r"\bthreat intelligence\b",
    r"\bmalware analysis\b",
    r"\breverse engineering\b",
    r"\bdigital forensics\b",
    r"\bdfir\b",
    r"\bsecurity research(?:er|ers)?\b",
    r"\bvulnerability disclosure\b",
    r"\bweb security\b",
    r"\bcloud security\b",
    r"\bnetwork security\b",
    r"\bhacking\b",
    r"\bhacker(?:s)?\b",
]

SOFT_KEYWORD_RE = re.compile("|".join(SOFT_KEYWORD_PATTERNS), flags=re.IGNORECASE)
URL_ONLY_RE = re.compile(r"^\s*(https?://\S+)(\s+https?://\S+)*\s*$", flags=re.IGNORECASE)
SPAN_RE = re.compile(r"</?span[^>]*>", flags=re.IGNORECASE)
HTML_RE = re.compile(r"<[^>]+>")
URL_RE = re.compile(r"https?://\S+", flags=re.IGNORECASE)
NON_WORD_RE = re.compile(r"[^\w\s-]", flags=re.UNICODE)
MULTISPACE_RE = re.compile(r"\s+")

KEYWORD_PATTERNS[:10]

['\\bmalware\\b',
 '\\bransomware\\b',
 '\\bphishing\\b',
 '\\bcredential(?:s)?\\b',
 '\\blogin\\b',
 '\\bexploit(?:ed|s|ing)?\\b',
 '\\bcve-\\d{4}-\\d+\\b',
 '\\bvulnerability\\b',
 '\\bzero[ -]?day\\b',
 '\\bbreach\\b']

In [9]:
def strip_accents(text: str) -> str:
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch)
    )


def normalize_message(text: str) -> str:
    text = SPAN_RE.sub(" ", text)
    text = HTML_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = strip_accents(text.lower())
    text = NON_WORD_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text)
    return text.strip()


def is_url_only(text: str) -> bool:
    return bool(URL_ONLY_RE.fullmatch(text or ""))


def keyword_hits(normalized_text: str) -> list[str]:
    return sorted(set(match.group(0) for match in KEYWORD_RE.finditer(normalized_text)))


def soft_keyword_hits(normalized_text: str) -> list[str]:
    return sorted(set(match.group(0) for match in SOFT_KEYWORD_RE.finditer(normalized_text)))


def parse_post_datetime(post: dict[str, Any]) -> datetime | None:
    value = post.get("datetime_utc")
    if isinstance(value, str) and value.strip():
        try:
            return datetime.fromisoformat(value.replace("Z", "+00:00")).astimezone(timezone.utc)
        except ValueError:
            pass

    value = post.get("timestamp")
    if isinstance(value, (int, float)):
        return datetime.fromtimestamp(value, tz=timezone.utc)

    value = post.get("date")
    if isinstance(value, str) and value.strip():
        for fmt in ("%Y-%m-%d", "%Y-%m-%d %H:%M:%S"):
            try:
                return datetime.strptime(value[:19], fmt).replace(tzinfo=timezone.utc)
            except ValueError:
                continue
    return None


def ensure_temporal_fields(post: dict[str, Any]) -> dict[str, Any]:
    enriched = dict(post)
    parsed = parse_post_datetime(enriched)
    if parsed is not None:
        enriched["datetime_utc"] = parsed.isoformat()
        enriched["timestamp"] = int(parsed.timestamp())
        enriched["date"] = parsed.strftime("%Y-%m-%d")
    return enriched


def evaluate_post(post: dict[str, Any]) -> dict[str, Any]:
    raw_message = str(post.get("message", "")).strip()
    normalized = normalize_message(raw_message)
    hits = keyword_hits(normalized)
    soft_hits = soft_keyword_hits(normalized)
    char_count = len(normalized)
    token_count = len(normalized.split()) if normalized else 0

    if is_url_only(raw_message):
        decision = "drop"
        reason = "url_only"
    elif not normalized:
        decision = "drop"
        reason = "empty_after_normalization"
    elif hits and (char_count < MIN_CHARS or token_count < MIN_TOKENS):
        decision = "keep_review"
        reason = "short_but_keyword_hit"
    elif hits:
        decision = "keep"
        reason = "keyword_match"
    elif soft_hits and char_count >= MIN_CHARS and token_count >= MIN_TOKENS:
        decision = "keep_review"
        reason = "soft_keyword_match"
    else:
        decision = "drop"
        reason = "no_keyword_match"

    enriched = ensure_temporal_fields(post)
    enriched["cleaning"] = {
        "normalized_message": normalized,
        "keyword_hits": hits,
        "soft_keyword_hits": soft_hits,
        "char_count": char_count,
        "token_count": token_count,
        "decision": decision,
        "reason": reason,
    }
    return enriched


def should_keep(post: dict[str, Any]) -> bool:
    return post["cleaning"]["decision"] in {"keep", "keep_review"}

In [10]:
def process_file(json_path: Path) -> tuple[Path, int, int]:
    with json_path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    if not isinstance(data, list):
        raise ValueError(f"{json_path} nao contem uma lista JSON na raiz.")
    if not data:
        raise ValueError(f"{json_path} nao contem posts para avaliar.")

    items_to_process = data[:1] if SAMPLE_ONE_PER_FILE else data
    evaluated = []

    for item in tqdm(items_to_process, desc=json_path.name, leave=False):
        evaluated.append(evaluate_post(item))

    kept = [item for item in evaluated if should_keep(item)]
    output_path = OUTPUT_DIR / json_path.name
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(kept, file, ensure_ascii=False, indent=2)
        file.write("\n")

    return output_path, len(evaluated), len(kept)


results = []
for json_file in tqdm(JSON_FILES, desc="Arquivos"):
    with json_file.open("r", encoding="utf-8") as file:
        payload = json.load(file)
    if isinstance(payload, list) and len(payload) > 0:
        results.append(process_file(json_file))

results[:5]

Arquivos: 100%|██████████| 14/14 [00:07<00:00,  1.97it/s]


[(PosixPath('../sentinel_replica_jsons_cleaned_eval/HackingBlogsGroup.json'),
  724,
  298),
 (PosixPath('../sentinel_replica_jsons_cleaned_eval/PHOfficial.json'),
  1618,
  340),
 (PosixPath('../sentinel_replica_jsons_cleaned_eval/WokeIntelDrops.json'),
  42,
  2),
 (PosixPath('../sentinel_replica_jsons_cleaned_eval/bellingcat.json'), 8, 0),
 (PosixPath('../sentinel_replica_jsons_cleaned_eval/cissp.json'), 2466, 1024)]

In [11]:
summary_rows = []
for output_path, total_evaluated, total_kept in results:
    source_path = INPUT_DIR / output_path.name
    with source_path.open("r", encoding="utf-8") as file:
        source_items = json.load(file)
    items = source_items[:1] if SAMPLE_ONE_PER_FILE else source_items
    for item in items:
        evaluated = evaluate_post(item)
        cleaning = evaluated["cleaning"]
        summary_rows.append(
            {
                "file": output_path.name,
                "id": evaluated.get("id"),
                "decision": cleaning["decision"],
                "reason": cleaning["reason"],
                "char_count": cleaning["char_count"],
                "token_count": cleaning["token_count"],
                "datetime_utc": evaluated.get("datetime_utc"),
                "timestamp": evaluated.get("timestamp"),
                "keyword_hits": ", ".join(cleaning["keyword_hits"]),
                "soft_keyword_hits": ", ".join(cleaning.get("soft_keyword_hits", [])),
                "message": evaluated.get("message", "")[:180],
            }
        )

summary_df = pd.DataFrame(summary_rows)
summary_df

,file,id,decision,reason,char_count,token_count,keyword_hits,message
0,HackingBlogsGroup.json,85e1007c-0678-45f9-b718-34100432a328,drop,url_only,0,0,,https://www.linkedin.com/posts/notifybreach_cy...
1,HackingBlogsGroup.json,60d5a521-3f2b-4afb-91d4-94f646633343,drop,no_keyword_match,1716,269,,🚨💀FREE NOTES API-HACKING DAY 3: Finding Anyone...
2,HackingBlogsGroup.json,de9562f6-81dc-4df7-9250-d406b1b41a1a,keep,keyword_match,1804,284,compromise,🔥FREE NOTES API-HACKING BOOTCAMP DAY 1 : SETTI...
3,HackingBlogsGroup.json,f60d84ec-0770-42e6-a387-dd035f0bfae2,drop,no_keyword_match,1125,189,,🔥FREE NOTES API-HACKING DAY2 : Introduction An...
4,HackingBlogsGroup.json,f80e9944-6e76-4d28-ba13-94752755e410,keep,keyword_match,1361,221,"leaked, ransomware",🚨🚨A Secret Hacker GangExposed Is Exposing the ...
...,...,...,...,...,...,...,...,...
268940,joinhackingarmy.json,461584e2-554b-417f-b309-a7fab7e98875,keep,keyword_match,603,95,phishing,**🧬 How To Hack AnyBody's Camera And IP Just B...
268941,joinhackingarmy.json,cd1b4a19-368f-4ebb-8cfc-d1b4351d3f64,drop,no_keyword_match,344,58,,**➡️Learn Python from Basics➡️\n\nZip course c...
268942,joinhackingarmy.json,244f8cea-dd3c-49ca-bf04-bef0d2a48533,drop,no_keyword_match,14,3,,Class wala pdf
268943,joinhackingarmy.json,659adcac-a3ed-4921-b14d-60defcf5939d,drop,no_keyword_match,41,9,,**7 Bje Tyaar Rahe Sab**. **Aaj Live Stream Ho...


In [12]:
summary_df["decision"].value_counts(dropna=False)

decision
drop           259013
keep             9168
keep_review       764
Name: count, dtype: int64